<a href="https://colab.research.google.com/github/Leanhchudang2511/baitaptrituenhantao/blob/main/AppDuDoan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏠 TÍNH TIỀN PHÒNG TRỌ - PHIÊN BẢN XỊN XÒ v2.0
### Giao diện đẹp | Đăng nhập | Hamburger Menu | Dark/Light Mode | AI dự đoán

---
**Chạy từng cell theo thứ tự từ trên xuống dưới**

1. **Cell 1**: Cài thư viện
2. **Cell 2**: Upload ảnh nền (tùy chọn)
3. **Cell 3**: Viết file `app.py`
4. **Cell 4**: Khởi động với ngrok → nhận link public

In [1]:

!pip install streamlit pyngrok scikit-learn pandas openpyxl matplotlib seaborn --quiet
print('✅ Cài đặt xong!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 43.4 MB/s eta 0:00:00
✅ Cài đặt xong!


In [2]:
import base64, os
try:
    from google.colab import files
    print('📸 Upload ảnh nền cho trang đăng nhập (tùy chọn).')
    print('   Nhấn Cancel nếu muốn dùng nền gradient mặc định.\n')
    uploaded = files.upload()
    for fname in uploaded:
        ext = fname.split('.')[-1].lower()
        dest = f'bg_image.{ext}'
        if fname != dest:
            os.rename(fname, dest)
        with open(dest, 'rb') as f:
            b64 = base64.b64encode(f.read()).decode()
        with open('bg_b64.txt', 'w') as f:
            f.write(f'data:image/{ext};base64,{b64}')
        print(f'✅ Đã encode ảnh nền xong!')
except Exception as e:
    print(f'⏭️ Bỏ qua upload - dùng nền gradient mặc định. ({e})')

📸 Upload ảnh nền cho trang đăng nhập (tùy chọn).
   Nhấn Cancel nếu muốn dùng nền gradient mặc định.



Saving giadienphongtro.xlsx to giadienphongtro.xlsx
✅ Đã encode ảnh nền xong!


In [3]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pickle, os, json, hashlib, warnings
from datetime import datetime
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
warnings.filterwarnings('ignore')

EXCEL_FILE = 'giadienphongtro.xlsx'
USER_FILE  = 'users.json'

st.set_page_config(
    page_title='Tính Tiền Phòng Trọ AI',
    page_icon='🏠',
    layout='wide',
    initial_sidebar_state='collapsed'
)

KHU_VUC = {
    'TP.HCM - Quận 1': 1.5,
    'TP.HCM - Quận 2 (Thủ Đức)': 1.35,
    'TP.HCM - Quận 3': 1.45,
    'TP.HCM - Quận 7 (Phú Mỹ Hưng)': 1.4,
    'TP.HCM - Bình Thạnh': 1.2,
    'TP.HCM - Tân Bình': 1.25,
    'TP.HCM - Gò Vấp': 1.1,
    'TP.HCM - Bình Chánh': 0.9,
    'TP.HCM - Hóc Môn': 0.85,
    'Hà Nội - Hoàn Kiếm': 1.45,
    'Hà Nội - Cầu Giấy': 1.3,
    'Hà Nội - Đống Đa': 1.35,
    'Hà Nội - Hà Đông': 1.0,
    'Đà Nẵng': 1.1,
    'Cần Thơ': 0.95,
    'Khác': 1.0,
}

# ── HELPERS ──────────────────────────────────────────────────
def hp(p):
    return hashlib.sha256(p.encode()).hexdigest()

def load_users():
    if os.path.exists(USER_FILE):
        with open(USER_FILE, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {'admin': hp('admin123')}

def save_users(u):
    with open(USER_FILE, 'w', encoding='utf-8') as f:
        json.dump(u, f, ensure_ascii=False)

def get_bg_css():
    if os.path.exists('bg_b64.txt'):
        with open('bg_b64.txt') as f:
            b64 = f.read().strip()
        return f'background-image: url("{b64}"); background-size: cover; background-position: center;'
    return 'background: linear-gradient(135deg, #0f1724 0%, #1a2a4a 40%, #0d3b6e 70%, #1565c0 100%);'

@st.cache_data
def load_data_cached():
    if os.path.exists(EXCEL_FILE):
        try:
            raw = pd.read_excel(EXCEL_FILE)
            if 'tien_dien' in raw.columns:
                return raw[raw['tien_dien'] > 0].dropna(subset=['tien_dien']).reset_index(drop=True)
        except:
            pass
    sample = {
        'loai_nguoi': ['Sinh viên', 'Hộ gia đình'] * 10,
        'so_nguoi':   [2,3,1,4,3,5,2,5,3,2,2,2,2,2,4,5,5,2,2,1],
        'dien_tich':  [23,50,25,53,25,78,20,85,46,101,20,85,25,54,79,114,95,26,17,26],
        'tang':       [3,9,4,7,4,4,4,0,19,9,4,8,4,19,1,9,14,0,2,3],
        'loai_hinh':  ['Ký túc xá','Căn hộ','Phòng trọ','Căn hộ','Phòng trọ',
                       'Nhà phố','Phòng trọ','Căn hộ','Căn hộ','Căn hộ',
                       'Phòng trọ','Nhà phố','Căn hộ dịch vụ','Căn hộ','Nhà phố',
                       'Căn hộ','Nhà phố','Phòng trọ','Phòng trọ','Phòng trọ'],
        'co_tu_lanh': [1] * 20,
        'so_quat':      [2,2,1,3,1,3,2,3,2,3,2,3,2,3,3,4,2,1,2,2],
        'so_may_lanh':  [1,3,0,1,1,1,1,3,3,1,1,2,1,2,1,1,1,1,0,0],
        'gio_may_lanh': [4,7,0,13,7,7,13,6,7,10,13,9,11,9,8,5,13,9,0,0],
        'tien_dien':    [952000,2052000,448000,1478000,1300000,935000,2240000,2590000,
                         2481000,1722000,2240000,1885000,1843000,2590000,1494000,
                         1050000,1939000,1516000,552000,564000],
        'gia_dien':     [3500,2500,3800,2500,3800,2500,4000,3500,3000,3500,
                         4000,2500,3800,3500,3500,3000,3500,3800,4000,4000],
        'khu_vuc':      ['TP.HCM - Tân Bình'] * 20,
        'thang':        [datetime.now().strftime('%Y-%m')] * 20,
    }
    df = pd.DataFrame(sample)
    df.to_excel(EXCEL_FILE, index=False)
    return df

def get_features(df, df_ref):
    d = df.copy()
    all_nguoi = list(df_ref['loai_nguoi'].unique())
    all_hinh  = list(df_ref['loai_hinh'].unique())
    le_n = LabelEncoder().fit(all_nguoi)
    le_h = LabelEncoder().fit(all_hinh)
    d['loai_nguoi_enc'] = le_n.transform(
        [x if x in all_nguoi else all_nguoi[0] for x in d['loai_nguoi'].astype(str)])
    d['loai_hinh_enc'] = le_h.transform(
        [x if x in all_hinh else all_hinh[0] for x in d['loai_hinh'].astype(str)])
    d['kwh_ml']    = d['so_may_lanh'] * d['gio_may_lanh'] * 1.5 * 30
    d['kwh_quat']  = d['so_quat'] * 0.075 * 8 * 30
    d['kwh_tl']    = d['co_tu_lanh'] * 0.1 * 24 * 30
    d['kwh_total'] = d['kwh_ml'] + d['kwh_quat'] + d['kwh_tl'] + 20
    return d[['loai_nguoi_enc','so_nguoi','dien_tich','tang','loai_hinh_enc',
              'co_tu_lanh','so_quat','so_may_lanh','gio_may_lanh','gia_dien',
              'kwh_ml','kwh_quat','kwh_tl','kwh_total']]

@st.cache_resource
def train_models():
    if os.path.exists('model.pkl'):
        try:
            with open('model.pkl', 'rb') as f:
                d = pickle.load(f)
            return d['model'], d['results']
        except:
            pass
    df = load_data_cached()
    X  = get_features(df, df)
    y  = df['tien_dien']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    mdls = {
        'Random Forest':     RandomForestRegressor(n_estimators=150, random_state=42),
        'Gradient Boosting': GradientBoostingRegressor(n_estimators=150, random_state=42),
        'Ridge Regression':  Ridge(alpha=1.0),
    }
    results = {}
    best = None
    best_r2 = -999
    for name, m in mdls.items():
        m.fit(X_tr, y_tr)
        pred = m.predict(X_te)
        r2   = r2_score(y_te, pred)
        mae  = mean_absolute_error(y_te, pred)
        rmse = np.sqrt(mean_squared_error(y_te, pred))
        results[name] = {'model': m, 'r2': r2, 'mae': mae, 'rmse': rmse}
        if r2 > best_r2:
            best_r2 = r2
            best = m
    return best, results

# ── SESSION STATE ────────────────────────────────────────────
defaults = {
    'logged_in': False,
    'username': '',
    'dark_mode': True,
    'page': 'login',
    'auth_tab': 'login',
    'df': None,
}
for k, v in defaults.items():
    if k not in st.session_state:
        st.session_state[k] = v
if st.session_state.df is None:
    st.session_state.df = load_data_cached()

best_model, model_results = train_models()
users_db = load_users()

# ── THEME ────────────────────────────────────────────────────
dark  = st.session_state.dark_mode
BG    = '#0f1724'   if dark else '#f0f4f8'
CARD  = '#1a2744'   if dark else '#ffffff'
TEXT  = '#e8edf5'   if dark else '#1a2744'
SUB   = '#8ba3c7'   if dark else '#5a7299'
ACC   = '#4f8ef7'
ACC2  = '#00d4aa'
BORDER = '#2a3f6f'  if dark else '#d0daea'
FC    = '#1a2744'   if dark else '#ffffff'
TC    = '#8ba3c7'   if dark else '#5a7299'
bg_css = get_bg_css()

# ── MASTER CSS ───────────────────────────────────────────────
st.markdown(f"""
<style>
@import url('https://fonts.googleapis.com/css2?family=Be+Vietnam+Pro:wght@300;400;500;600;700;800;900&family=JetBrains+Mono:wght@400;600&display=swap');

*, *::before, *::after {{ box-sizing: border-box; }}

html, body, [data-testid="stAppViewContainer"], .main {{
    background: {BG} !important;
    font-family: 'Be Vietnam Pro', sans-serif !important;
    color: {TEXT} !important;
}}
[data-testid="stHeader"] {{ display: none !important; }}
.block-container {{ padding: 0 !important; max-width: 100% !important; }}

/* TOPBAR */
.topbar {{
    position: fixed; top: 0; left: 0; right: 0; z-index: 9999;
    height: 60px;
    background: {'rgba(15,23,36,0.96)' if dark else 'rgba(255,255,255,0.96)'};
    backdrop-filter: blur(20px);
    border-bottom: 1px solid {BORDER};
    display: flex; align-items: center; justify-content: space-between;
    padding: 0 24px;
    box-shadow: 0 2px 20px rgba(0,0,0,0.25);
}}
.topbar-brand {{
    font-size: 16px; font-weight: 800; color: {TEXT};
    display: flex; align-items: center; gap: 8px;
}}
.topbar-brand span {{ color: {ACC}; }}
.topbar-right {{ display: flex; align-items: center; gap: 12px; }}
.badge-user {{
    background: linear-gradient(135deg, {ACC}, {ACC2});
    color: white; font-size: 12px; font-weight: 700;
    padding: 4px 14px; border-radius: 20px;
}}

/* MAIN CONTENT */
.main-content {{ padding-top: 72px; min-height: 100vh; background: {BG}; }}
.page-header {{ padding: 28px 36px 12px; }}
.page-header h2 {{ font-size: 26px; font-weight: 900; margin-bottom: 4px; color: {TEXT}; }}
.page-header p {{ color: {SUB}; font-size: 13px; }}
.page-body {{ padding: 0 36px 36px; }}

/* LOGIN PAGE */
.login-page {{
    min-height: 100vh;
    {bg_css}
    display: flex; align-items: center; justify-content: center;
    position: relative;
}}
.login-overlay {{
    position: absolute; inset: 0;
    background: rgba(0,0,0,0.6);
    backdrop-filter: blur(3px);
}}
.login-card {{
    position: relative; z-index: 10;
    background: rgba(8,15,30,0.9);
    border: 1px solid rgba(79,142,247,0.3);
    border-radius: 24px;
    padding: 52px 48px;
    width: 100%; max-width: 440px;
    backdrop-filter: blur(30px);
    box-shadow: 0 40px 80px rgba(0,0,0,0.6), 0 0 0 1px rgba(79,142,247,0.1);
}}
.login-logo {{
    width: 76px; height: 76px; border-radius: 22px;
    background: linear-gradient(135deg, {ACC}, {ACC2});
    display: flex; align-items: center; justify-content: center;
    font-size: 38px; margin: 0 auto 22px;
    box-shadow: 0 12px 36px rgba(79,142,247,0.45);
}}
.login-title {{
    text-align: center; color: white;
    font-size: 22px; font-weight: 900;
    letter-spacing: -0.5px; margin-bottom: 6px;
}}
.login-sub {{
    text-align: center;
    color: rgba(255,255,255,0.5);
    font-size: 12px; margin-bottom: 32px;
}}
.login-hint {{
    text-align: center;
    color: rgba(255,255,255,0.35);
    font-size: 11px; margin-top: 16px;
}}
.login-hint b {{ color: {ACC}; }}

/* INPUTS */
[data-testid="stTextInput"] input {{
    background: rgba(255,255,255,0.06) !important;
    border: 1px solid rgba(255,255,255,0.12) !important;
    border-radius: 12px !important;
    color: white !important;
    font-family: 'Be Vietnam Pro', sans-serif !important;
    font-size: 14px !important;
}}
[data-testid="stTextInput"] input:focus {{
    border-color: {ACC} !important;
    box-shadow: 0 0 0 3px rgba(79,142,247,0.2) !important;
}}
[data-testid="stTextInput"] label {{
    color: rgba(255,255,255,0.65) !important;
    font-size: 13px !important; font-weight: 600 !important;
    font-family: 'Be Vietnam Pro', sans-serif !important;
}}

/* BUTTONS */
[data-testid="stButton"] button {{
    font-family: 'Be Vietnam Pro', sans-serif !important;
    font-weight: 700 !important;
    border-radius: 12px !important;
    transition: all 0.2s !important;
}}
[data-testid="stButton"] button[kind="primary"] {{
    background: linear-gradient(135deg, {ACC}, #3a7df5) !important;
    border: none !important; color: white !important;
    box-shadow: 0 4px 20px rgba(79,142,247,0.4) !important;
}}
[data-testid="stButton"] button[kind="primary"]:hover {{
    transform: translateY(-1px) !important;
    box-shadow: 0 8px 28px rgba(79,142,247,0.5) !important;
}}

/* METRIC */
[data-testid="stMetric"] {{
    background: {CARD} !important;
    border: 1px solid {BORDER} !important;
    border-radius: 14px !important;
    padding: 18px !important;
}}
[data-testid="stMetricLabel"] {{ color: {SUB} !important; font-size: 12px !important; }}
[data-testid="stMetricValue"] {{
    color: {TEXT} !important;
    font-family: 'JetBrains Mono', monospace !important;
}}

/* RESULT BOX */
.result-box {{
    background: {'rgba(79,142,247,0.08)' if dark else 'rgba(79,142,247,0.05)'};
    border: 1px solid rgba(79,142,247,0.3);
    border-radius: 16px; padding: 24px; margin-top: 20px;
}}
.result-amount {{
    font-size: 40px; font-weight: 900;
    background: linear-gradient(135deg, {ACC}, {ACC2});
    -webkit-background-clip: text; -webkit-text-fill-color: transparent;
    font-family: 'JetBrains Mono', monospace; line-height: 1.2;
}}

/* SIDEBAR */
[data-testid="stSidebar"] {{
    background: {'#0d1829' if dark else '#f8faff'} !important;
    border-right: 1px solid {BORDER} !important;
}}
[data-testid="stSidebar"] * {{ color: {TEXT} !important; }}
[data-testid="stSidebar"] [data-testid="stButton"] button {{
    background: {'rgba(255,255,255,0.04)' if dark else 'rgba(0,0,0,0.04)'} !important;
    border: 1px solid {BORDER} !important;
    color: {TEXT} !important; text-align: left !important;
    justify-content: flex-start !important;
}}
[data-testid="stSidebar"] [data-testid="stButton"] button[kind="primary"] {{
    background: linear-gradient(135deg, {ACC}, #3a7df5) !important;
    border: none !important; color: white !important;
}}

/* MISC */
p, label, span {{ color: {TEXT} !important; }}
h1, h2, h3 {{ color: {TEXT} !important; font-family: 'Be Vietnam Pro' !important; }}
.stAlert {{ border-radius: 12px !important; }}
[data-testid="stDataFrame"] {{ border-radius: 12px !important; overflow: hidden; }}
::-webkit-scrollbar {{ width: 5px; height: 5px; }}
::-webkit-scrollbar-thumb {{ background: {BORDER}; border-radius: 5px; }}
</style>
""", unsafe_allow_html=True)

# ================================================================
#  TRANG ĐĂNG NHẬP
# ================================================================
if not st.session_state.logged_in:
    st.markdown(f"""
    <div class="login-page">
      <div class="login-overlay"></div>
      <div class="login-card">
        <div class="login-logo">🏠</div>
        <div class="login-title">TÍNH TIỀN PHÒNG TRỌ</div>
        <div class="login-sub">Quản lý dễ dàng – Thu tiền chính xác – AI dự đoán</div>
      </div>
    </div>
    """, unsafe_allow_html=True)

    _, col, _ = st.columns([1, 1, 1])
    with col:
        st.markdown('<div style="height:72px"></div>', unsafe_allow_html=True)

        t1, t2 = st.columns(2)
        with t1:
            if st.button('🔑  Đăng nhập', use_container_width=True,
                         type='primary' if st.session_state.auth_tab == 'login' else 'secondary',
                         key='tab_login'):
                st.session_state.auth_tab = 'login'
                st.rerun()
        with t2:
            if st.button('📝  Đăng ký', use_container_width=True,
                         type='primary' if st.session_state.auth_tab == 'register' else 'secondary',
                         key='tab_reg'):
                st.session_state.auth_tab = 'register'
                st.rerun()

        st.markdown('<div style="height:12px"></div>', unsafe_allow_html=True)

        if st.session_state.auth_tab == 'login':
            lu = st.text_input('👤  Tên đăng nhập', placeholder='Nhập tên đăng nhập...', key='lu')
            lp = st.text_input('🔒  Mật khẩu', type='password', placeholder='Nhập mật khẩu...', key='lp')
            st.markdown('<div style="height:4px"></div>', unsafe_allow_html=True)
            if st.button('🚀  ĐĂNG NHẬP', type='primary', use_container_width=True, key='btn_login'):
                if lu in users_db and users_db[lu] == hp(lp):
                    st.session_state.logged_in = True
                    st.session_state.username  = lu
                    st.session_state.page      = 'dashboard'
                    st.rerun()
                else:
                    st.error('❌ Sai tên đăng nhập hoặc mật khẩu!')
            st.markdown(f'<div class="login-hint">Demo: <b>admin</b> / <b>admin123</b></div>',
                        unsafe_allow_html=True)
        else:
            ru = st.text_input('👤  Tên đăng nhập mới', placeholder='Chọn tên đăng nhập...', key='ru')
            rp = st.text_input('🔒  Mật khẩu (tối thiểu 6 ký tự)', type='password', key='rp')
            rc = st.text_input('🔒  Xác nhận mật khẩu', type='password', key='rc')
            if st.button('✅  ĐĂNG KÝ', type='primary', use_container_width=True, key='btn_reg'):
                if not ru or not rp:
                    st.error('Vui lòng nhập đầy đủ!')
                elif len(rp) < 6:
                    st.error('Mật khẩu phải từ 6 ký tự trở lên!')
                elif rp != rc:
                    st.error('Mật khẩu không khớp!')
                elif ru in users_db:
                    st.error('Tên đã tồn tại, chọn tên khác!')
                else:
                    users_db[ru] = hp(rp)
                    save_users(users_db)
                    st.success(f'🎉 Đăng ký thành công! Chào mừng {ru}')
                    st.session_state.auth_tab = 'login'
                    st.rerun()
    st.stop()

# ================================================================
#  TOPBAR (sau đăng nhập)
# ================================================================
st.markdown(f"""
<div class="topbar">
  <div class="topbar-brand">🏠 Tính Tiền <span>Phòng Trọ</span></div>
  <div class="topbar-right">
    <div class="badge-user">👤 {st.session_state.username}</div>
    <div style="color:{SUB};font-size:12px">{datetime.now().strftime("%d/%m/%Y")}</div>
  </div>
</div>
""", unsafe_allow_html=True)

# ── SIDEBAR MENU ─────────────────────────────────────────────
with st.sidebar:
    st.markdown(f"""<div style='padding:16px 0 8px'>
        <div style='font-size:18px;font-weight:900;margin-bottom:2px'>☰ Menu</div>
        <div style='font-size:12px;color:{SUB}'>Xin chào, <b>{st.session_state.username}</b></div>
    </div>""", unsafe_allow_html=True)
    st.markdown('---')

    pages_list = [
        ('dashboard', '📊 Dashboard'),
        ('predict',   '🤖 Dự đoán AI'),
        ('stats',     '📈 Thống kê'),
        ('data',      '💾 Dữ liệu'),
        ('areas',     '📍 Khu vực'),
        ('ml',        '🧠 Hiệu suất ML'),
    ]
    for pid, label in pages_list:
        is_active = st.session_state.page == pid
        if st.button(label, key=f'nav_{pid}', use_container_width=True,
                     type='primary' if is_active else 'secondary'):
            st.session_state.page = pid
            st.rerun()

    st.markdown('---')
    st.markdown(f"<div style='font-size:11px;font-weight:700;color:{SUB};letter-spacing:1px;margin-bottom:8px'>⚙️ CÀI ĐẶT</div>",
                unsafe_allow_html=True)
    new_dark = st.toggle('🌙 Chế độ tối', value=dark, key='dark_toggle')
    if new_dark != dark:
        st.session_state.dark_mode = new_dark
        st.rerun()

    st.markdown('---')
    if st.button('🚪 Đăng xuất', use_container_width=True, key='logout_btn'):
        st.session_state.logged_in = False
        st.session_state.username  = ''
        st.session_state.page      = 'login'
        st.rerun()
    st.markdown(f"<div style='font-size:10px;color:{SUB};margin-top:8px;text-align:center'>v2.0 Xịn Xò Edition ✨</div>",
                unsafe_allow_html=True)

# ── CONTENT WRAPPER ──────────────────────────────────────────
st.markdown('<div class="main-content">', unsafe_allow_html=True)
page   = st.session_state.page
df_ref = st.session_state.df

# ================================================================
#  DASHBOARD
# ================================================================
if page == 'dashboard':
    st.markdown(f"""
    <div class="page-header">
      <h2>📊 Dashboard Tổng Quan</h2>
      <p>Xin chào <b>{st.session_state.username}</b>! {datetime.now().strftime("%A, %d/%m/%Y")}</p>
    </div>
    """, unsafe_allow_html=True)

    with st.container():
        st.markdown('<div class="page-body">', unsafe_allow_html=True)

        c1, c2, c3, c4 = st.columns(4)
        c1.metric('📋 Tổng bản ghi',  len(df_ref))
        c2.metric('💰 TB Tiền điện',  f"{df_ref['tien_dien'].mean()/1000:,.0f}k đ")
        c3.metric('📐 TB Diện tích',  f"{df_ref['dien_tich'].mean():,.0f} m²")
        c4.metric('👥 TB Số người',   f"{df_ref['so_nguoi'].mean():,.1f}")

        st.markdown('<div style="height:20px"></div>', unsafe_allow_html=True)

        col_a, col_b = st.columns(2)
        with col_a:
            st.markdown(f'<div style="font-size:14px;font-weight:700;margin-bottom:10px">🏢 Phân bố loại hình</div>',
                        unsafe_allow_html=True)
            lh_counts = df_ref['loai_hinh'].value_counts()
            colors = ['#4f8ef7','#00d4aa','#f7934f','#f74f7a','#a64ff7']
            fig1, ax1 = plt.subplots(figsize=(5, 3.5))
            fig1.patch.set_facecolor(FC); ax1.set_facecolor(FC)
            wedges, texts, autotexts = ax1.pie(
                lh_counts.values, labels=lh_counts.index,
                autopct='%1.0f%%', colors=colors[:len(lh_counts)],
                startangle=90, pctdistance=0.75
            )
            for t in texts:
                t.set_color(TC); t.set_fontsize(8)
            for t in autotexts:
                t.set_color('white'); t.set_fontsize(8); t.set_fontweight('bold')
            plt.tight_layout()
            st.pyplot(fig1); plt.close(fig1)

        with col_b:
            st.markdown(f'<div style="font-size:14px;font-weight:700;margin-bottom:10px">📈 TB Tiền điện theo khu vực</div>',
                        unsafe_allow_html=True)
            by_kv = df_ref.groupby('khu_vuc')['tien_dien'].mean().sort_values(ascending=True).tail(8)
            fig2, ax2 = plt.subplots(figsize=(5, 3.5))
            fig2.patch.set_facecolor(FC); ax2.set_facecolor(FC)
            bars2 = ax2.barh(by_kv.index, by_kv.values / 1000, color='#4f8ef7', alpha=0.85)
            ax2.bar_label(bars2, fmt='%.0fk', padding=4, color=TC, fontsize=8)
            ax2.set_xlabel('Nghìn VNĐ', color=TC, fontsize=9)
            ax2.tick_params(colors=TC, labelsize=8)
            for sp in ax2.spines.values(): sp.set_visible(False)
            plt.tight_layout()
            st.pyplot(fig2); plt.close(fig2)

        st.markdown('<div style="height:12px"></div>', unsafe_allow_html=True)
        st.markdown(f'<div style="font-size:14px;font-weight:700;margin-bottom:10px">⚡ Truy cập nhanh</div>',
                    unsafe_allow_html=True)
        qa1, qa2, qa3, qa4 = st.columns(4)
        with qa1:
            if st.button('🤖 Dự đoán AI', use_container_width=True, type='primary', key='qa1'):
                st.session_state.page = 'predict'; st.rerun()
        with qa2:
            if st.button('📈 Thống kê', use_container_width=True, key='qa2'):
                st.session_state.page = 'stats'; st.rerun()
        with qa3:
            if st.button('💾 Dữ liệu', use_container_width=True, key='qa3'):
                st.session_state.page = 'data'; st.rerun()
        with qa4:
            if st.button('🧠 Hiệu suất ML', use_container_width=True, key='qa4'):
                st.session_state.page = 'ml'; st.rerun()

        st.markdown('</div>', unsafe_allow_html=True)

# ================================================================
#  DỰ ĐOÁN AI
# ================================================================
elif page == 'predict':
    st.markdown(f"""
    <div class="page-header">
      <h2>🤖 Dự Đoán Tiền Điện bằng AI</h2>
      <p>Nhập thông tin phòng → AI dự đoán ngay lập tức → Tự động lưu Excel</p>
    </div>
    """, unsafe_allow_html=True)

    with st.container():
        st.markdown('<div class="page-body">', unsafe_allow_html=True)

        c1, c2, c3 = st.columns(3)
        with c1:
            st.markdown(f'<div style="font-size:13px;font-weight:700;color:{SUB};margin-bottom:10px">👥 THÔNG TIN NGƯỜI Ở</div>',
                        unsafe_allow_html=True)
            p_ln = st.radio('Loại người ở', ['Sinh viên', 'Hộ gia đình'], key='p_ln')
            p_sn = st.slider('Số người ở', 1, 10, 2, key='p_sn')
            p_kv = st.selectbox('Khu vực', list(KHU_VUC.keys()), index=5, key='p_kv')
            p_gd = st.number_input('Giá điện (đ/kWh)', value=3800, step=100, key='p_gd')
        with c2:
            st.markdown(f'<div style="font-size:13px;font-weight:700;color:{SUB};margin-bottom:10px">🏠 THÔNG TIN PHÒNG</div>',
                        unsafe_allow_html=True)
            p_dt = st.slider('Diện tích (m²)', 10, 200, 25, key='p_dt')
            p_tg = st.slider('Tầng số', 0, 30, 3, key='p_tg')
            p_lh = st.selectbox('Loại hình',
                ['Phòng trọ','Căn hộ','Căn hộ dịch vụ','Nhà phố','Ký túc xá'], key='p_lh')
        with c3:
            st.markdown(f'<div style="font-size:13px;font-weight:700;color:{SUB};margin-bottom:10px">⚡ THIẾT BỊ ĐIỆN</div>',
                        unsafe_allow_html=True)
            p_tl = st.radio('Tủ lạnh', ['Có', 'Không'], key='p_tl')
            p_sq = st.slider('Số quạt', 0, 6, 2, key='p_sq')
            p_sm = st.slider('Số máy lạnh', 0, 5, 1, key='p_sm')
            p_gm = st.slider('Giờ bật máy lạnh/ngày', 0.0, 24.0, 8.0, 0.5, key='p_gm')

        cb1, cb2 = st.columns(2)
        with cb1:
            predict_btn = st.button('⚡ DỰ ĐOÁN TIỀN ĐIỆN', type='primary',
                                    use_container_width=True, key='btn_predict')
        with cb2:
            retrain_btn = st.button('🔄 Huấn luyện lại ML',
                                    use_container_width=True, key='btn_retrain')

        if predict_btn:
            try:
                tl      = 1 if p_tl == 'Có' else 0
                kv_mult = KHU_VUC.get(p_kv, 1.0)
                row = pd.DataFrame([{
                    'loai_nguoi': p_ln, 'so_nguoi': p_sn,
                    'dien_tich': float(p_dt), 'tang': p_tg,
                    'loai_hinh': p_lh, 'co_tu_lanh': tl,
                    'so_quat': p_sq, 'so_may_lanh': p_sm,
                    'gio_may_lanh': float(p_gm), 'gia_dien': float(p_gd),
                }])
                Xrow      = get_features(row, df_ref)
                pred_kv   = best_model.predict(Xrow)[0] * kv_mult
                kwh_total = Xrow['kwh_total'].values[0]
                kwh_ml_v  = Xrow['kwh_ml'].values[0]
                kwh_q     = Xrow['kwh_quat'].values[0]
                kwh_tl_v  = Xrow['kwh_tl'].values[0]
                tay       = kwh_total * p_gd

                m1, m2, m3, m4 = st.columns(4)
                m1.metric('🤖 Dự đoán ML',      f'{pred_kv:,.0f} đ')
                m2.metric('🧮 Tính tay',         f'{tay:,.0f} đ')
                m3.metric('⚡ kWh ước tính',     f'{kwh_total:,.1f}')
                m4.metric('📍 Hệ số khu vực',    f'x{kv_mult:.2f}')

                st.markdown(f"""
                <div class="result-box">
                  <div style="font-size:12px;color:{SUB};margin-bottom:8px">💡 DỰ ĐOÁN AI THÁNG NÀY</div>
                  <div class="result-amount">{pred_kv:,.0f} đ</div>
                  <div style="margin-top:18px;padding-top:16px;border-top:1px solid rgba(79,142,247,0.2)">
                    <div style="font-size:12px;font-weight:700;color:{TEXT};margin-bottom:8px">Chi tiết tiêu thụ điện:</div>
                    <div style="font-size:13px;color:{SUB};line-height:1.8">
                      🌬️ Máy lạnh ({p_sm} cái × {p_gm}h/ngày): <b style="color:{TEXT}">{kwh_ml_v:,.1f} kWh</b><br>
                      💨 Quạt ({p_sq} cái × 8h/ngày): <b style="color:{TEXT}">{kwh_q:,.1f} kWh</b><br>
                      ❄️ Tủ lạnh (24h/ngày): <b style="color:{TEXT}">{kwh_tl_v:,.1f} kWh</b><br>
                      🔌 Thiết bị khác: <b style="color:{TEXT}">20.0 kWh</b>
                    </div>
                  </div>
                </div>
                """, unsafe_allow_html=True)

                new_row = {
                    'loai_nguoi': p_ln, 'so_nguoi': p_sn,
                    'dien_tich': float(p_dt), 'tang': p_tg,
                    'loai_hinh': p_lh, 'co_tu_lanh': tl,
                    'so_quat': p_sq, 'so_may_lanh': p_sm,
                    'gio_may_lanh': float(p_gm), 'tien_dien': pred_kv,
                    'gia_dien': float(p_gd), 'khu_vuc': p_kv,
                    'thang': datetime.now().strftime('%Y-%m'),
                }
                st.session_state.df = pd.concat(
                    [df_ref, pd.DataFrame([new_row])], ignore_index=True)
                st.session_state.df.to_excel(EXCEL_FILE, index=False)
                st.caption(f"✅ Đã lưu | Tổng: {len(st.session_state.df)} bản ghi")
            except Exception as e:
                st.error(f'Lỗi: {e}')

        if retrain_btn:
            st.cache_resource.clear()
            st.success('🔄 Đã xóa cache – tải lại trang để huấn luyện lại!')
            st.rerun()

        st.markdown('</div>', unsafe_allow_html=True)

# ================================================================
#  THỐNG KÊ
# ================================================================
elif page == 'stats':
    st.markdown('<div class="page-header"><h2>📈 Thống Kê & Phân Tích</h2></div>',
                unsafe_allow_html=True)
    with st.container():
        st.markdown('<div class="page-body">', unsafe_allow_html=True)
        df = st.session_state.df

        c1, c2, c3, c4 = st.columns(4)
        c1.metric('Tổng bản ghi', len(df))
        c2.metric('TB tiền điện', f"{df['tien_dien'].mean():,.0f} đ")
        c3.metric('TB diện tích', f"{df['dien_tich'].mean():,.1f} m²")
        c4.metric('TB số người',  f"{df['so_nguoi'].mean():,.1f}")

        st.markdown('<div style="height:20px"></div>', unsafe_allow_html=True)

        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        fig.patch.set_facecolor(FC)
        for ax in axes.flat:
            ax.set_facecolor(FC)
            ax.tick_params(colors=TC)
            for sp in ax.spines.values():
                sp.set_color('#2a3f6f' if dark else '#e8e8e8')

        axes[0,0].hist(df['tien_dien'] / 1000, bins=15, color='#4f8ef7', edgecolor='none', alpha=0.85)
        axes[0,0].set_title('Phân bố Tiền điện', color=TC, fontsize=11)
        axes[0,0].set_xlabel('Nghìn VNĐ', color=TC)

        lh_list = df['loai_hinh'].unique()
        clrs = ['#4f8ef7','#00d4aa','#f7934f','#f74f7a','#a64ff7']
        bp = axes[0,1].boxplot(
            [df[df['loai_hinh'] == lh]['tien_dien'].values / 1000 for lh in lh_list],
            labels=lh_list, patch_artist=True
        )
        for p, c in zip(bp['boxes'], clrs[:len(lh_list)]):
            p.set_facecolor(c); p.set_alpha(0.8)
        for key in ['whiskers','caps','medians','fliers']:
            for el in bp[key]: el.set_color(TC)
        axes[0,1].set_title('Theo Loại hình', color=TC, fontsize=11)
        axes[0,1].tick_params(axis='x', rotation=20, labelsize=7)

        for ln, col in [('Sinh viên','#4f8ef7'), ('Hộ gia đình','#f74f7a')]:
            m = df['loai_nguoi'] == ln
            if m.sum() > 0:
                axes[1,0].scatter(df[m]['dien_tich'], df[m]['tien_dien'] / 1000,
                                  c=col, label=ln, alpha=0.7, s=50)
        axes[1,0].set_title('Diện tích vs Tiền điện', color=TC, fontsize=11)
        axes[1,0].set_xlabel('m²', color=TC)
        leg = axes[1,0].legend()
        leg.get_frame().set_facecolor(FC)
        for t in leg.get_texts(): t.set_color(TC)

        by_kv = df.groupby('khu_vuc')['tien_dien'].mean().sort_values(ascending=False).head(8)
        axes[1,1].barh(by_kv.index, by_kv.values / 1000, color='#4f8ef7', alpha=0.85)
        axes[1,1].set_title('TB Tiền điện theo Khu vực', color=TC, fontsize=11)
        axes[1,1].set_xlabel('Nghìn VNĐ', color=TC)
        axes[1,1].tick_params(labelsize=7)

        fig.suptitle('Phân Tích Tiền Điện Phòng Trọ', color=TEXT,
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        st.pyplot(fig); plt.close(fig)
        st.markdown('</div>', unsafe_allow_html=True)

# ================================================================
#  DỮ LIỆU
# ================================================================
elif page == 'data':
    st.markdown('<div class="page-header"><h2>💾 Quản Lý Dữ Liệu</h2></div>',
                unsafe_allow_html=True)
    with st.container():
        st.markdown('<div class="page-body">', unsafe_allow_html=True)
        df_show = st.session_state.df
        st.caption(f"Tổng: {len(df_show)} bản ghi | {datetime.now().strftime('%H:%M %d/%m/%Y')}")
        st.dataframe(df_show, use_container_width=True, height=420)

        c_dl, c_up = st.columns(2)
        with c_dl:
            buf = df_show.to_csv(index=False).encode('utf-8-sig')
            st.download_button('📥 Xuất CSV', buf, 'giadienphongtro.csv',
                               'text/csv', use_container_width=True)
        with c_up:
            uploaded = st.file_uploader('📤 Upload Excel thực tế', type=['xlsx'])
            if uploaded:
                with open(EXCEL_FILE, 'wb') as f:
                    f.write(uploaded.getbuffer())
                st.cache_data.clear()
                st.success('✅ Đã upload! Tải lại để áp dụng.')
                st.rerun()
        st.markdown('</div>', unsafe_allow_html=True)

# ================================================================
#  KHU VỰC
# ================================================================
elif page == 'areas':
    st.markdown('<div class="page-header"><h2>📍 Hệ Số Khu Vực</h2></div>',
                unsafe_allow_html=True)
    with st.container():
        st.markdown('<div class="page-body">', unsafe_allow_html=True)
        kv_df = pd.DataFrame.from_dict(KHU_VUC, orient='index', columns=['Hệ số'])
        kv_df.index.name = 'Khu vực'
        kv_df = kv_df.sort_values('Hệ số', ascending=False)
        st.dataframe(kv_df.style.background_gradient(cmap='Blues'), use_container_width=True)

        fig2, ax2 = plt.subplots(figsize=(10, 5))
        fig2.patch.set_facecolor(FC); ax2.set_facecolor(FC)
        ax2.barh(kv_df.index, kv_df['Hệ số'], color='#4f8ef7', alpha=0.85)
        ax2.axvline(1.0, color='#f74f7a', linestyle='--', label='Chuẩn = 1.0')
        ax2.set_xlabel('Hệ số nhân', color=TC)
        ax2.set_title('Hệ số giá điện theo khu vực', color=TEXT, fontweight='bold')
        ax2.tick_params(colors=TC, labelsize=9)
        for sp in ax2.spines.values():
            sp.set_color('#2a3f6f' if dark else '#e8e8e8')
        leg2 = ax2.legend()
        leg2.get_frame().set_facecolor(FC)
        for t in leg2.get_texts(): t.set_color(TC)
        plt.tight_layout()
        st.pyplot(fig2); plt.close(fig2)
        st.markdown('</div>', unsafe_allow_html=True)

elif page == 'ml':
    st.markdown('<div class="page-header"><h2>🧠 Hiệu Suất Machine Learning</h2></div>',
                unsafe_allow_html=True)
    with st.container():
        st.markdown('<div class="page-body">', unsafe_allow_html=True)
        rows = []
        for name, res in model_results.items():
            rows.append({
                'Mô hình':  name,
                'R² Score': f"{res['r2']:.4f}",
                'MAE (đ)':  f"{res['mae']:,.0f}",
                'RMSE (đ)': f"{res['rmse']:,.0f}",
            })
        st.dataframe(pd.DataFrame(rows), use_container_width=True, hide_index=True)

        st.markdown(f"""
        <div class="result-box" style="margin-top:16px">
          <div style="font-weight:700;margin-bottom:10px">📊 Đánh giá chất lượng mô hình</div>
          <div style="font-size:13px;color:{SUB};line-height:2">
            🟢 R² ≥ 0.9 → Rất tốt &nbsp;|&nbsp;
            🟡 R² 0.7–0.9 → Tốt &nbsp;|&nbsp;
            🔴 R² &lt; 0.7 → Cần thêm dữ liệu
          </div>
        </div>
        """, unsafe_allow_html=True)

        st.info(f"📊 Dữ liệu hiện tại: **{len(st.session_state.df)}** bản ghi. "
                f"Thêm dữ liệu thực tế → Huấn luyện lại → Mô hình tốt hơn!")

        names = list(model_results.keys())
        r2s   = [model_results[n]['r2'] for n in names]
        fig3, ax3 = plt.subplots(figsize=(8, 3.5))
        fig3.patch.set_facecolor(FC); ax3.set_facecolor(FC)
        bars3 = ax3.bar(names, r2s,
                        color=['#4f8ef7','#00d4aa','#f7934f'], alpha=0.85, width=0.5)
        ax3.bar_label(bars3, fmt='%.4f', padding=4, color=TC, fontsize=10, fontweight='bold')
        ax3.axhline(0.9, color='#00d4aa', linestyle='--', alpha=0.5, label='Ngưỡng tốt (0.9)')
        ax3.set_ylim(0, 1.08)
        ax3.set_ylabel('R² Score', color=TC)
        ax3.set_title('So sánh R² các mô hình ML', color=TEXT, fontweight='bold')
        ax3.tick_params(colors=TC)
        for sp in ax3.spines.values():
            sp.set_color('#2a3f6f' if dark else '#e8e8e8')
        leg3 = ax3.legend()
        leg3.get_frame().set_facecolor(FC)
        for t in leg3.get_texts(): t.set_color(TC)
        plt.tight_layout()
        st.pyplot(fig3); plt.close(fig3)
        st.markdown('</div>', unsafe_allow_html=True)

st.markdown('</div>', unsafe_allow_html=True)

Writing app.py


In [4]:
NGROK_TOKEN = '3DFvCJVYwIGdAyqwiLEDuaBONez_29crAqnArhG8sbtsvXpC3'

import subprocess, time, threading
from pyngrok import ngrok, conf

subprocess.run(['pkill', '-9', '-f', 'streamlit'], capture_output=True)
subprocess.run(['pkill', '-9', '-f', 'ngrok'],     capture_output=True)
time.sleep(3)

conf.get_default().auth_token = NGROK_TOKEN

def run_streamlit():
    subprocess.run([
        'streamlit', 'run', 'app.py',
        '--server.port', '8501',
        '--server.headless', 'true',
        '--server.enableCORS', 'false',
        '--server.enableXsrfProtection', 'false',
    ])

threading.Thread(target=run_streamlit, daemon=True).start()

print('⏳ Đang khởi động Streamlit...')
time.sleep(7)

public_url = ngrok.connect(8501)
print()
print('=' * 58)
print('🚀 APP XỊN XÒ ĐÃ KHỞI ĐỘNG THÀNH CÔNG!')
print('=' * 58)
print(f'🌐 PUBLIC LINK: {public_url}')
print()
print('📱 Copy link → mở trên điện thoại hoặc máy khác!')
print('🔑 Demo login: admin / admin123')
print('☰  Bấm 3 gạch góc trái để mở menu')
print('⚠️  Link còn hoạt động khi cell này đang chạy.')
print('=' * 58)

⏳ Đang khởi động Streamlit...

🚀 APP XỊN XÒ ĐÃ KHỞI ĐỘNG THÀNH CÔNG!
🌐 PUBLIC LINK: NgrokTunnel: "https://bruising-slideshow-chapped.ngrok-free.dev" -> "http://localhost:8501"

📱 Copy link → mở trên điện thoại hoặc máy khác!
🔑 Demo login: admin / admin123
☰  Bấm 3 gạch góc trái để mở menu
⚠️  Link còn hoạt động khi cell này đang chạy.
